# Geneformer model-size sweep on PBMC — MI curves

One line per architecture trial, x = quality (log), y = MI per signal. Counterpart to the STATE
model-sizing notebook (`2026-04-20_14-31_plotting_state_model_sizing_pbmc.ipynb`) but swept axis
is now Geneformer BERT body parameter count (`num_embed_dim` / `num_layers`) instead of
STATE's transformer sizes. Loss curves come from HF Trainer's `trainer_state.json`
(HuggingFace `log_history`), not Lightning's `metrics.csv`.

In [ ]:
# Gather results by walking the actual on-disk layout:
#   model_sizing_geneformer/model_sizing_geneformer_XX/<size>/<quality>/
#     ├─ config.json                                   (arch + fixed HPs + metadata; written pre-train)
#     ├─ result.json                                   (post-run; status=ok|error, train_time_s)
#     ├─ trainer_state.json                            (final HF log_history: train loss + eval_loss)
#     ├─ checkpoint-*/trainer_state.json               (per-checkpoint snapshots; latest = full history)
#     └─ MI/<seed>/Y_<signal>_<quality>_geneformer/lmi_mutual_information.txt
#
# The model_sizing_geneformer_XX.yaml index files are only written once the whole sweep finishes,
# so we walk model_sizing_geneformer_* directly and read config.json for the arch dict.
# Per-step training and per-eval validation loss curves come from HF Trainer's log_history.
# The resulting DataFrame holds scalars; the raw curves live in the CURVES dict.
import json
import re
from pathlib import Path

import pandas as pd

OUTPUT_DIR = Path('/home/igor/noise_scaling/data/other/model_sizing_geneformer')
MI_SEED = 42  # first seed emitted by latentmi
ARCH_KEYS = ('num_embed_dim', 'intermed_size', 'num_attn_heads', 'num_layers', 'max_input_size')

def _num(s: str):
    try: return int(s)
    except ValueError:
        try: return float(s)
        except ValueError: return s

# Token vocabulary size for PBMC (loaded once from the utils pickle so we can
# instantiate exact BERT models and count parameters). If the pickle is
# missing we fall back to a conservative default and emit a warning.
try:
    import pickle
    TOKEN_DICT_PATH = Path('/home/igor/noise_scaling/data/PBMC/utils/token_dict.pkl')
    with open(TOKEN_DICT_PATH, 'rb') as fp:
        _token_dict = pickle.load(fp)
    VOCAB_SIZE = len(_token_dict)
    PAD_TOKEN_ID = _token_dict.get('<pad>', 0)
except Exception as e:
    print(f'WARNING: could not load token_dict.pkl ({e}); using VOCAB_SIZE=20000')
    VOCAB_SIZE, PAD_TOKEN_ID = 20000, 0

# Compute trainable-param counts ONCE per unique Geneformer arch by
# instantiating BertForMaskedLM with the matching config and summing tensor
# numels. Much cheaper than analytical bookkeeping of every BERT tensor.
# Cached in _PARAM_COUNTS so repeated calls are free.
_PARAM_COUNTS: dict = {}
def _trainable_params(arch: dict) -> int:
    key = tuple(int(arch.get(k, 0) or 0) for k in ARCH_KEYS)
    if key in _PARAM_COUNTS:
        return _PARAM_COUNTS[key]
    from transformers import BertConfig, BertForMaskedLM
    hidden, intermed, heads, layers, max_pos = key
    if hidden <= 0 or layers <= 0:
        _PARAM_COUNTS[key] = 0
        return 0
    cfg = BertConfig(
        hidden_size=hidden, num_hidden_layers=layers, num_attention_heads=heads,
        intermediate_size=intermed, max_position_embeddings=max_pos,
        vocab_size=VOCAB_SIZE, pad_token_id=PAD_TOKEN_ID, hidden_act='relu',
        attention_probs_dropout_prob=0.02, hidden_dropout_prob=0.02,
        layer_norm_eps=1e-12, initializer_range=0.02,
    )
    m = BertForMaskedLM(cfg)
    n = sum(p.numel() for p in m.parameters() if p.requires_grad)
    del m
    _PARAM_COUNTS[key] = int(n)
    return int(n)

def _read_hf_log_history(log_history: list[dict]):
    """Split HF Trainer log_history into (train_df, val_df).

    train_df: columns [step, train_loss]; val_df: columns [step, val_loss].
    HF convention: train entries have 'loss', eval entries have 'eval_loss';
    both have 'step'. We drop rows missing the relevant key and sort by step.
    """
    if not log_history:
        return None, None
    tr = [{'step': e['step'], 'train_loss': e['loss']}
          for e in log_history if 'loss' in e and 'step' in e and 'eval_loss' not in e]
    va = [{'step': e['step'], 'val_loss': e['eval_loss']}
          for e in log_history if 'eval_loss' in e and 'step' in e]
    tr_df = pd.DataFrame(tr).sort_values('step').reset_index(drop=True) if tr else None
    va_df = pd.DataFrame(va).sort_values('step').reset_index(drop=True) if va else None
    return tr_df, va_df

def _read_trainer_state(leaf: Path):
    """Return (train_df, val_df) from the most complete trainer_state.json.

    Prefers leaf/trainer_state.json (written by `trainer.save_model` at end
    of training). Falls back to the highest-numbered checkpoint-*/trainer_state.json
    (which HF writes after every save; the latest checkpoint holds the
    whole log_history so far).
    """
    candidates: list[Path] = []
    final = leaf / 'trainer_state.json'
    if final.exists():
        candidates.append(final)
    ckpt_dirs = sorted(
        (p for p in leaf.glob('checkpoint-*') if p.is_dir()),
        key=lambda p: int(p.name.split('-', 1)[1]) if p.name.split('-', 1)[1].isdigit() else -1,
    )
    if ckpt_dirs:
        candidates.append(ckpt_dirs[-1] / 'trainer_state.json')
    for path in candidates:
        if not path.exists() or path.stat().st_size == 0:
            continue
        try:
            state = json.loads(path.read_text())
        except Exception:
            continue
        tr, va = _read_hf_log_history(state.get('log_history', []))
        if (tr is not None and len(tr)) or (va is not None and len(va)):
            return tr, va
    return None, None

rows = []
CURVES: dict = {}  # (trial_id, size, quality) -> {'train': DataFrame, 'val': DataFrame}

for trial_dir in sorted(OUTPUT_DIR.glob('model_sizing_geneformer_*')):
    if not trial_dir.is_dir():
        continue
    m = re.match(r'model_sizing_geneformer_(\d+)$', trial_dir.name)
    if not m:
        continue
    trial_id = int(m.group(1))

    for size_dir in sorted(p for p in trial_dir.iterdir() if p.is_dir()):
        size_val = _num(size_dir.name)
        if not isinstance(size_val, (int, float)):
            continue
        for q_dir in sorted(p for p in size_dir.iterdir() if p.is_dir()):
            quality_val = _num(q_dir.name)
            if not isinstance(quality_val, (int, float)):
                continue

            cfg_path = q_dir / 'config.json'
            cfg = json.loads(cfg_path.read_text()) if cfg_path.exists() else {}
            arch = cfg.get('arch') or {}
            row = {
                'trial_id': trial_id,
                'trial_name': cfg.get('trial_name', f'model_sizing_geneformer_{trial_id:02d}'),
                'size': int(size_val),
                'quality': float(quality_val),
                **{k: arch.get(k) for k in ARCH_KEYS},
                'trainable_params': _trainable_params(arch) if arch else 0,
            }

            res_path = q_dir / 'result.json'
            if res_path.exists():
                try:
                    res = json.loads(res_path.read_text())
                    row['status'] = res.get('status', 'unknown')
                    row['train_time_s'] = res.get('train_time_s')
                except json.JSONDecodeError:
                    row['status'] = 'corrupt'
            else:
                row['status'] = 'pending'

            # HF training / eval curves from trainer_state.json.
            # Final values go into df; full curves live in CURVES for plotting.
            train_df, val_df = _read_trainer_state(q_dir)
            if train_df is not None and len(train_df):
                row['train_loss_final'] = float(train_df['train_loss'].iloc[-1])
                row['n_train_steps'] = int(train_df['step'].iloc[-1])
            if val_df is not None and len(val_df):
                row['val_loss_final'] = float(val_df['val_loss'].iloc[-1])
            CURVES[(trial_id, int(size_val), float(quality_val))] = {
                'train': train_df, 'val': val_df,
            }

            mi_root = q_dir / 'MI' / str(MI_SEED)
            if mi_root.is_dir():
                for sig_dir in filter(Path.is_dir, mi_root.iterdir()):
                    # Y_<signal>_<quality-as-text>_geneformer  ->  strip quality + tag.
                    sm = re.match(r'Y_(.+?)_[0-9][0-9_.]*(?:_geneformer)?$', sig_dir.name)
                    signal = sm.group(1) if sm else sig_dir.name
                    f = sig_dir / 'lmi_mutual_information.txt'
                    if f.exists():
                        try: row[f'mi_{signal}'] = float(f.read_text().strip())
                        except ValueError: pass
            rows.append(row)

df = pd.DataFrame(rows)
if len(df):
    df = df.sort_values(['trial_id', 'size', 'quality']).reset_index(drop=True)
mi_cols_found = sorted(c for c in df.columns if c.startswith('mi_'))
n_trials = df['trial_id'].nunique() if len(df) else 0
n_ok = int((df['status'] == 'ok').sum()) if 'status' in df.columns else 0
n_curves = sum(1 for v in CURVES.values() if v['train'] is not None and len(v['train']))
print(f'{len(df)} rows across {n_trials} trials  ok={n_ok}  curves={n_curves}  mi_cols={mi_cols_found}')
df

In [ ]:
# Completion matrix: rows = model-size trials (5), cols = qualities (10).
# Cell = status from result.json ('ok' / 'error' / 'pending' / 'corrupt' / missing).
# Bottom rows summarise per-quality totals; right-most column summarises per-trial totals.
EXPECTED_TRIALS = 5
EXPECTED_QUALITIES = 10

if not len(df):
    print('df is empty — nothing to tally.')
else:
    status_mat = df.pivot_table(index='trial_id', columns='quality',
                                values='status', aggfunc='first').sort_index()
    # Fill any (trial, quality) combinations we never walked (directory absent) as 'missing'.
    status_mat = status_mat.fillna('missing')

    trial_done    = status_mat.apply(lambda r: (r == 'ok').sum(), axis=1)
    trial_missing = status_mat.shape[1] - trial_done
    quality_done    = status_mat.apply(lambda c: (c == 'ok').sum(), axis=0)
    quality_missing = status_mat.shape[0] - quality_done

    display_mat = status_mat.copy()
    display_mat['done']    = trial_done
    display_mat['missing'] = trial_missing

    summary = pd.DataFrame(
        [list(quality_done) + [quality_done.sum(), quality_missing.sum()],
         list(quality_missing) + ['', '']],
        index=['done', 'missing'], columns=display_mat.columns,
    )
    full = pd.concat([display_mat, summary])

    total_cells   = status_mat.size
    total_done    = int((status_mat == 'ok').sum().sum())
    total_missing = total_cells - total_done
    print(f'trials seen: {status_mat.shape[0]} / expected {EXPECTED_TRIALS}   '
          f'qualities seen: {status_mat.shape[1]} / expected {EXPECTED_QUALITIES}')
    print(f'runs done (status=ok): {total_done} / {total_cells}   missing: {total_missing}')
    full

In [ ]:
# Train / eval loss curves on linear axes: 2 columns (train, eval), 1 row per quality.
# Each subplot overlays one line per model-size trial (using the curves stashed in CURVES) plus
# the original-work Geneformer training curve at the same (size, quality) as a bold black
# dashed line — pulled from data/PBMC/<size>/<quality>/results/Geneformer/<latest-checkpoint>/trainer_state.json.
#
# ONE of the sweep trials is hardcoded to match the production Geneformer arch
# (scaling_laws.algo.geneformer.Geneformer defaults: num_embed_dim=256, num_layers=3,
# num_attn_heads=4, intermed_size=512) — see PRODUCTION_ARCH in the compute script.
# The matching sweep trial gets:
#   - '(replica of orig)' appended to its legend entry
#   - 2.2x linewidth so it stands out among the other trials
#   - an in-plot annotation ('replica of orig (MNN)') at the end of its curve in each subplot
# The black-dashed 'orig: Geneformer' baseline is a SEPARATE prior run with the same arch —
# kept on the plot so run-to-run divergence (different step budget, different warmup duty-cycle)
# at the same arch is visible.
#
# Train curves are smoothed with a centered rolling mean if they contain enough logged points
# (>=100 points) — our sweep trials log every 10 optimizer steps so they benefit from
# smoothing; the orig production curve typically logs every 1000 steps and is plotted raw.
#
# Checkpoint marker: when early stopping triggered (best eval_loss is followed by >= ES_PATIENCE
# worse evals, matching the compute script's EarlyStoppingCallback), the eval curve gets an
# 'X' marker at (best_step, best_eval) — the checkpoint that would actually be used for
# downstream evaluation.
#
# Legend param count: total TRAINABLE params computed by instantiating BertForMaskedLM with
# the matching config (includes BERT embeddings + body + MLM head). Unlike the STATE sweep
# Geneformer has no frozen sub-module, so there's only one number to report.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patheffects import withStroke
from pathlib import Path

ORIG_GENEFORMER_ROOT = Path('/home/igor/noise_scaling/data/PBMC')
SKIP_STEPS = 20            # drop first N steps per curve — too noisy to plot usefully
TRAIN_SMOOTH_WINDOW = 200  # centered rolling-mean window for train loss (only if enough points)
SMOOTH_MIN_POINTS = 100    # below this, plot train raw (too sparse to smooth meaningfully)
ES_PATIENCE = 5            # matches FIXED_HPARAMS['early_stopping_patience'] in compute script

# Shared multi-line legend title used by every plot in this notebook.
PARAM_LEGEND_TITLE = (
    'format: M## h<hidden>/L<nlayers>\n'
    '(trainable M)\n'
    '  trainable = BertForMaskedLM total\n'
    '              (embeddings + body + head)\n'
    '  exact count from config.json arch'
)

# Production arch: scaling_laws.algo.geneformer.Geneformer defaults. Must stay in sync with
# the CONTROL trial in the compute script (2026-04-22_15-49_compute_geneformer_model_sizing_pbmc.py).
PRODUCTION_ARCH = {
    'num_embed_dim':  256,
    'intermed_size':  512,
    'num_attn_heads': 4,
    'num_layers':     3,
}
PROD_LINEWIDTH = 2.2

def _find_prod_trial(frame: pd.DataFrame) -> int | None:
    """Return the trial_id whose arch fields match PRODUCTION_ARCH exactly."""
    if frame.empty or not set(PRODUCTION_ARCH).issubset(frame.columns):
        return None
    mask = np.ones(len(frame), dtype=bool)
    for k, v in PRODUCTION_ARCH.items():
        col = pd.to_numeric(frame[k], errors='coerce').fillna(-1).astype(int).values
        mask &= (col == int(v))
    hits = frame.loc[mask, 'trial_id'].unique()
    return int(hits[0]) if len(hits) else None

def _orig_geneformer_curves(size: int, quality) -> tuple[pd.DataFrame | None, pd.DataFrame | None]:
    """Load (train_df, val_df) from the orig-work Geneformer run at (size, quality).

    Reads the highest-numbered checkpoint-N/trainer_state.json under
    data/PBMC/<size>/<quality>/results/Geneformer/. Reuses _read_trainer_state
    from the data-gathering cell above.
    """
    base = ORIG_GENEFORMER_ROOT / str(size) / str(quality) / 'results' / 'Geneformer'
    if not base.is_dir():
        return None, None
    return _read_trainer_state(base)

def _smooth_train(tr: pd.DataFrame | None) -> pd.DataFrame | None:
    """Drop first SKIP_STEPS and replace train_loss with a centered rolling mean.

    Smoothing is skipped entirely when the curve has fewer than SMOOTH_MIN_POINTS
    points (typical of production runs that log every 1000 steps — smoothing there
    would just average the curve out to a constant).
    """
    if tr is None or not len(tr):
        return tr
    tr = tr[tr['step'] >= SKIP_STEPS].copy()
    if not len(tr):
        return tr
    if len(tr) < SMOOTH_MIN_POINTS:
        return tr
    tr['train_loss'] = tr['train_loss'].rolling(
        window=TRAIN_SMOOTH_WINDOW, min_periods=10, center=True).mean()
    return tr.dropna(subset=['train_loss'])

def _detect_es(va: pd.DataFrame, patience: int = ES_PATIENCE) -> tuple[bool, int, float]:
    """Return (es_triggered, best_step, best_val) for an eval curve."""
    if va is None or va.empty:
        return False, 0, float('nan')
    bidx = va['val_loss'].idxmin()
    n_after = len(va) - 1 - va.index.get_loc(bidx)
    return (n_after >= patience, int(va.loc[bidx, 'step']), float(va.loc[bidx, 'val_loss']))

def _annotate_replica(ax, x: float, y: float, text: str, color) -> None:
    ax.annotate(
        text, xy=(x, y), xytext=(5, 0), textcoords='offset points',
        fontsize=7, color=color, va='center', ha='left', fontweight='bold',
        zorder=8, clip_on=False,
        path_effects=[withStroke(linewidth=2.0, foreground='white')],
    )

qualities = sorted(df['quality'].unique())
trial_ids = sorted(df['trial_id'].unique())
cmap = plt.get_cmap('viridis')
trial_color = {tid: cmap(i / max(1, len(trial_ids) - 1)) for i, tid in enumerate(trial_ids)}

prod_tid = _find_prod_trial(df)
if prod_tid is None:
    print('WARNING: no sweep trial matches PRODUCTION_ARCH — replica overlay will be skipped.')
else:
    print(f'Production-replica trial detected: trial_id={prod_tid} (model_sizing_geneformer_{prod_tid:02d})')

def _trial_label(tid: int) -> str:
    sub = df[df['trial_id'] == tid]
    if sub.empty:
        return f'M{tid:02d}'
    cfg = sub.iloc[0]
    tag = ' (replica of orig)' if tid == prod_tid else ''
    return (f"M{int(tid):02d} "
            f"h{int(cfg['num_embed_dim'])}/L{int(cfg['num_layers'])} "
            f"({cfg['trainable_params']/1e6:.2f}M){tag}")

trial_label = {tid: _trial_label(tid) for tid in trial_ids}
replica_note = f'replica of orig (M{prod_tid:02d})' if prod_tid is not None else ''

sizes = sorted({k[1] for k in CURVES}) or [100000]
orig_size = sizes[0]

ORIG_CURVES = {q: _orig_geneformer_curves(orig_size, q) for q in qualities}
n_orig_tr = sum(1 for tr, _ in ORIG_CURVES.values() if tr is not None and len(tr))
n_orig_va = sum(1 for _, va in ORIG_CURVES.values() if va is not None and len(va))
print(f'orig Geneformer curves loaded: train={n_orig_tr}/{len(qualities)} val={n_orig_va}/{len(qualities)}')

n_rows = len(qualities)
fig, axes = plt.subplots(n_rows, 2, figsize=(11, 2.4 * n_rows),
                         sharex='col', squeeze=False)

orig_drawn = False
es_marker_drawn = False
for r, q in enumerate(qualities):
    ax_tr, ax_va = axes[r, 0], axes[r, 1]
    for tid in trial_ids:
        is_prod = (tid == prod_tid)
        lw = PROD_LINEWIDTH if is_prod else 1.0
        keys = [k for k in CURVES if k[0] == tid and np.isclose(k[2], q)]
        for key in keys:
            curves = CURVES[key]
            tr = _smooth_train(curves['train'])
            va_full = curves['val']
            va = va_full[va_full['step'] >= SKIP_STEPS] if va_full is not None and len(va_full) else va_full
            color = trial_color[tid]
            if tr is not None and len(tr):
                ax_tr.plot(tr['step'].values, tr['train_loss'].values,
                           color=color, linewidth=lw, alpha=0.85,
                           zorder=4 if is_prod else 2)
                if is_prod and replica_note:
                    _annotate_replica(ax_tr,
                                      float(tr['step'].iloc[-1]),
                                      float(tr['train_loss'].iloc[-1]),
                                      replica_note, color)
            if va is not None and len(va):
                ax_va.plot(va['step'].values, va['val_loss'].values,
                           color=color, linewidth=lw, alpha=0.85,
                           marker='o', markersize=4 if is_prod else 3,
                           zorder=4 if is_prod else 2)
                if is_prod and replica_note:
                    _annotate_replica(ax_va,
                                      float(va['step'].iloc[-1]),
                                      float(va['val_loss'].iloc[-1]),
                                      replica_note, color)
                es, best_step, best_val = _detect_es(va_full)
                if es:
                    ax_va.scatter([best_step], [best_val],
                                  color=color, marker='X', s=65,
                                  edgecolor='k', linewidth=0.6, zorder=6)
                    es_marker_drawn = True

    otr, ova_full = ORIG_CURVES.get(q, (None, None))
    otr = _smooth_train(otr)
    ova = ova_full[ova_full['step'] >= SKIP_STEPS] if ova_full is not None and len(ova_full) else ova_full
    if otr is not None and len(otr):
        ax_tr.plot(otr['step'].values, otr['train_loss'].values,
                   color='k', linewidth=2.2, linestyle='--', zorder=5)
        orig_drawn = True
    if ova is not None and len(ova):
        ax_va.plot(ova['step'].values, ova['val_loss'].values,
                   color='k', linewidth=2.2, linestyle='--',
                   marker='s', markersize=4, zorder=5)
        orig_drawn = True
        es, best_step, best_val = _detect_es(ova_full)
        if es:
            ax_va.scatter([best_step], [best_val],
                          color='k', marker='X', s=90,
                          edgecolor='w', linewidth=0.8, zorder=7)
            es_marker_drawn = True

    ax_tr.set_ylabel(f'q={q:.4g}\nloss')
    ax_tr.grid(alpha=0.3)
    ax_va.grid(alpha=0.3)

axes[0, 0].set_title(f'train_loss (rolling mean, window={TRAIN_SMOOTH_WINDOW} steps, centered, when >{SMOOTH_MIN_POINTS} pts)')
axes[0, 1].set_title('eval_loss (per-eval, raw)')
axes[-1, 0].set_xlabel('step')
axes[-1, 1].set_xlabel('step')

legend_handles = [
    Line2D([0], [0], color=trial_color[tid],
           linewidth=PROD_LINEWIDTH if tid == prod_tid else 1.5,
           label=trial_label[tid])
    for tid in trial_ids
]
if orig_drawn:
    legend_handles.append(
        Line2D([0], [0], color='k', linewidth=2.2, linestyle='--',
               label='orig: Geneformer (data/PBMC/.../results/Geneformer)')
    )
if es_marker_drawn:
    legend_handles.append(
        Line2D([0], [0], color='gray', marker='X', linestyle='',
               markersize=9, markeredgecolor='k', markeredgewidth=0.6,
               label=f'checkpoint (ES, patience={ES_PATIENCE})')
    )
legend = fig.legend(handles=legend_handles, loc='center left', bbox_to_anchor=(1.0, 0.5),
                    fontsize=8, title=PARAM_LEGEND_TITLE, title_fontsize=8)
legend.get_title().set_multialignment('left')
fig.suptitle(
    f'Geneformer PBMC model-size sweep (size={orig_size}): train loss (centered rolling mean) '
    f'+ raw per-eval loss (X = early-stopping checkpoint, patience={ES_PATIENCE}), '
    f'first {SKIP_STEPS} steps dropped',
    y=1.0,
)
plt.tight_layout()
plt.show()

In [ ]:
# Same train/eval loss plot as above, but WITHOUT the in-plot 'replica of orig'
# text annotation. The replica trial is still rendered at PROD_LINEWIDTH and its
# legend entry still carries the '(replica of orig)' tag — only the inline label
# next to the curve endpoint is suppressed. All helper functions, CURVES, df,
# ORIG_CURVES, and PARAM_LEGEND_TITLE are reused from the cell above.
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D

n_rows = len(qualities)
fig, axes = plt.subplots(n_rows, 2, figsize=(11, 2.4 * n_rows),
                         sharex='col', squeeze=False)

orig_drawn = False
es_marker_drawn = False
for r, q in enumerate(qualities):
    ax_tr, ax_va = axes[r, 0], axes[r, 1]
    for tid in trial_ids:
        is_prod = (tid == prod_tid)
        lw = PROD_LINEWIDTH if is_prod else 1.0
        keys = [k for k in CURVES if k[0] == tid and np.isclose(k[2], q)]
        for key in keys:
            curves = CURVES[key]
            tr = _smooth_train(curves['train'])
            va_full = curves['val']
            va = va_full[va_full['step'] >= SKIP_STEPS] if va_full is not None and len(va_full) else va_full
            color = trial_color[tid]
            if tr is not None and len(tr):
                ax_tr.plot(tr['step'].values, tr['train_loss'].values,
                           color=color, linewidth=lw, alpha=0.85,
                           zorder=4 if is_prod else 2)
            if va is not None and len(va):
                ax_va.plot(va['step'].values, va['val_loss'].values,
                           color=color, linewidth=lw, alpha=0.85,
                           marker='o', markersize=4 if is_prod else 3,
                           zorder=4 if is_prod else 2)
                es, best_step, best_val = _detect_es(va_full)
                if es:
                    ax_va.scatter([best_step], [best_val],
                                  color=color, marker='X', s=65,
                                  edgecolor='k', linewidth=0.6, zorder=6)
                    es_marker_drawn = True

    otr, ova_full = ORIG_CURVES.get(q, (None, None))
    otr = _smooth_train(otr)
    ova = ova_full[ova_full['step'] >= SKIP_STEPS] if ova_full is not None and len(ova_full) else ova_full
    if otr is not None and len(otr):
        ax_tr.plot(otr['step'].values, otr['train_loss'].values,
                   color='k', linewidth=2.2, linestyle='--', zorder=5)
        orig_drawn = True
    if ova is not None and len(ova):
        ax_va.plot(ova['step'].values, ova['val_loss'].values,
                   color='k', linewidth=2.2, linestyle='--',
                   marker='s', markersize=4, zorder=5)
        orig_drawn = True
        es, best_step, best_val = _detect_es(ova_full)
        if es:
            ax_va.scatter([best_step], [best_val],
                          color='k', marker='X', s=90,
                          edgecolor='w', linewidth=0.8, zorder=7)
            es_marker_drawn = True

    ax_tr.set_ylabel(f'q={q:.4g}\nloss')
    ax_tr.grid(alpha=0.3)
    ax_va.grid(alpha=0.3)

axes[0, 0].set_title(f'train_loss (rolling mean, window={TRAIN_SMOOTH_WINDOW} steps, centered, when >{SMOOTH_MIN_POINTS} pts)')
axes[0, 1].set_title('eval_loss (per-eval, raw)')
axes[-1, 0].set_xlabel('step')
axes[-1, 1].set_xlabel('step')

legend_handles = [
    Line2D([0], [0], color=trial_color[tid],
           linewidth=PROD_LINEWIDTH if tid == prod_tid else 1.5,
           label=trial_label[tid])
    for tid in trial_ids
]
if orig_drawn:
    legend_handles.append(
        Line2D([0], [0], color='k', linewidth=2.2, linestyle='--',
               label='orig: Geneformer (data/PBMC/.../results/Geneformer)')
    )
if es_marker_drawn:
    legend_handles.append(
        Line2D([0], [0], color='gray', marker='X', linestyle='',
               markersize=9, markeredgecolor='k', markeredgewidth=0.6,
               label=f'checkpoint (ES, patience={ES_PATIENCE})')
    )
legend = fig.legend(handles=legend_handles, loc='center left', bbox_to_anchor=(1.0, 0.5),
                    fontsize=8, title=PARAM_LEGEND_TITLE, title_fontsize=8)
legend.get_title().set_multialignment('left')
fig.suptitle(
    f'Geneformer PBMC model-size sweep (size={orig_size}): train loss (centered rolling mean) '
    f'+ raw per-eval loss (X = early-stopping checkpoint, patience={ES_PATIENCE}), '
    f'first {SKIP_STEPS} steps dropped — no inline replica label',
    y=1.0,
)
plt.tight_layout()
plt.show()

In [ ]:
# Original-work baselines: load MI for the per-algorithm runs that live alongside
# the Geneformer model-size sweep. Layout:
#   data/PBMC/<size>/<quality>/results/<algo>/model/MI/<seed>/Y_<signal>_<quality>[_<algo_tag>]/lmi_mutual_information.txt
# Geneformer tags its signal dirs with '_geneformer'; others don't.
import re
from pathlib import Path

import pandas as pd

ORIG_ROOT = Path('/home/igor/noise_scaling/data/PBMC')
ORIG_SIZE = 100000
ORIG_SEED = 42
ORIG_SIGNAL = 'protein_counts'  # match mi_col in the model-size MI plot below

# Y_<signal>_<quality>[_<algo_tag>]  — allow an optional trailing alpha suffix (e.g. '_geneformer').
SIG_RE = re.compile(r'Y_(.+?)_[0-9][0-9_.]*(?:_[A-Za-z]+)?$')

orig_rows = []
size_dir = ORIG_ROOT / str(ORIG_SIZE)
for q_dir in sorted(p for p in size_dir.iterdir() if p.is_dir()):
    try:
        quality = float(q_dir.name)
    except ValueError:
        continue
    results_dir = q_dir / 'results'
    if not results_dir.is_dir():
        continue
    for algo_dir in sorted(p for p in results_dir.iterdir() if p.is_dir()):
        algo = algo_dir.name
        if algo in ('TunableState', 'TunableGeneformer'):
            continue  # sweep variants, not production baselines
        mi_root = algo_dir / 'model' / 'MI' / str(ORIG_SEED)
        if not mi_root.is_dir():
            continue
        for sig_dir in mi_root.iterdir():
            if not sig_dir.is_dir():
                continue
            m = SIG_RE.match(sig_dir.name)
            signal = m.group(1) if m else sig_dir.name
            if signal != ORIG_SIGNAL:
                continue
            f = sig_dir / 'lmi_mutual_information.txt'
            if not f.exists():
                continue
            try:
                mi_val = float(f.read_text().strip())
            except ValueError:
                continue
            orig_rows.append({'algo': algo, 'quality': quality, 'mi': mi_val})

orig_df = pd.DataFrame(orig_rows).sort_values(['algo', 'quality']).reset_index(drop=True)
print(f'{len(orig_df)} original-work points across {orig_df["algo"].nunique()} algos: {sorted(orig_df["algo"].unique())}')
orig_df

In [ ]:
# Overlay, split into two panels (shared y-axis):
#   Left:  model-size sweep MI curves (one line per arch trial) + orig Geneformer baseline.
#   Right: all original-work baselines (Geneformer + every other algorithm).
# Legend param count: trainable BertForMaskedLM total (embeddings + body + head).
# Same multi-line title as the loss-curves plot so the two legends are consistent
# (reuses PARAM_LEGEND_TITLE from the loss-curves cell above).
import matplotlib.pyplot as plt

mi_col = 'mi_protein_counts'
fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(13.5, 4.8), sharey=True)

algo_colors = {
    'State':            'k',
    'Geneformer':       'tab:red',
    'SCVI':             'tab:green',
    'PCA':              'tab:purple',
    'RandomProjection': 'tab:gray',
}

# --- Left: model-size sweep (thin, viridis by trial) + original Geneformer (dashed) ---
trial_ids = sorted(df['trial_id'].unique())
cmap = plt.get_cmap('viridis')
trial_color = {tid: cmap(i / max(1, len(trial_ids) - 1)) for i, tid in enumerate(trial_ids)}
for tid in trial_ids:
    sub = df[(df['trial_id'] == tid) & df[mi_col].notna()].sort_values('quality')
    if sub.empty:
        continue
    cfg = sub.iloc[0]
    label = (f"M{int(tid):02d} "
             f"h{int(cfg['num_embed_dim'])}/L{int(cfg['num_layers'])} "
             f"({cfg['trainable_params']/1e6:.2f}M)")
    ax_l.plot(sub['quality'].values, sub[mi_col].values,
              marker='o', linewidth=1.0, alpha=0.85,
              color=trial_color[tid], label=label)

gf_sub = orig_df[orig_df['algo'] == 'Geneformer'].sort_values('quality')
if not gf_sub.empty:
    ax_l.plot(gf_sub['quality'].values, gf_sub['mi'].values,
              marker='s', linewidth=2.5, linestyle='--',
              color=algo_colors['Geneformer'], label='orig: Geneformer', zorder=5)

ax_l.set_xscale('log')
ax_l.set_xlabel('quality (downsampling factor)')
ax_l.set_ylabel(f'MI({ORIG_SIGNAL})  [nats]')
ax_l.set_title('Model-size sweep (thin) vs orig Geneformer (dashed)')
ax_l.grid(alpha=0.3)
legend_l = ax_l.legend(loc='center left', bbox_to_anchor=(1.02, 0.5),
                       fontsize=7, ncol=1,
                       title=PARAM_LEGEND_TITLE, title_fontsize=7)
legend_l.get_title().set_multialignment('left')

# --- Right: all original-work baselines (Geneformer + every other algo) ---
for algo, sub in orig_df.groupby('algo'):
    sub = sub.sort_values('quality')
    ax_r.plot(sub['quality'].values, sub['mi'].values,
              marker='s', linewidth=2.5, linestyle='--',
              color=algo_colors.get(algo, None),
              label=f'orig: {algo}', zorder=5)

ax_r.set_xscale('log')
ax_r.set_xlabel('quality (downsampling factor)')
ax_r.set_title('Original-work baselines (all algos)')
ax_r.grid(alpha=0.3)
ax_r.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=8, ncol=1)

fig.suptitle(f'PBMC size={ORIG_SIZE}: Geneformer model-size sweep vs original-work baselines')
plt.tight_layout()
plt.show()

In [ ]:
# Box plot: distribution of mi_protein_counts across model-size trials, one box per quality.
# x-axis is true log scale (positions = quality, ax.set_xscale('log')).
import matplotlib.pyplot as plt
import numpy as np

mi_col = 'mi_protein_counts'
if mi_col not in df.columns:
    print(f'{mi_col} not found — nothing to plot.')
else:
    sub = df[df[mi_col].notna()]
    qualities = sorted(sub['quality'].unique())
    data = [sub.loc[sub['quality'] == q, mi_col].values for q in qualities]
    n_per = [len(d) for d in data]

    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    positions = np.array(qualities, dtype=float)
    # On a log axis, a constant linear `width` would look microscopic at small x
    # and huge at large x. Pick a constant log-space half-width (≈12%) and emit
    # asymmetric widths in linear units that map to it.
    w = 0.12
    widths = positions * (np.exp(w) - np.exp(-w)) / 2
    ax.boxplot(data, positions=positions, widths=widths,
               showmeans=True, meanline=True,
               medianprops=dict(color='C0'),
               meanprops=dict(color='C3', linestyle='--'))
    for pos, d in zip(positions, data):
        ax.scatter(np.full_like(d, pos, dtype=float), d,
                   s=12, alpha=0.5, color='k', zorder=3)

    ax.set_xscale('log')
    ax.set_xticks(positions)
    ax.set_xticklabels([f'{q:g}\n(n={n})' for q, n in zip(qualities, n_per)],
                       fontsize=8)
    ax.set_xlabel('quality (downsampling factor, log scale)')
    ax.set_ylabel('mi_protein_counts')
    ax.set_title('MI(protein_counts) distribution across model-size trials, per quality')
    ax.grid(alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()